# Qwen3-TTS renderer v5 (jelly-jlpt): every missing clip, every level, one notebook

Renders **every** clip the input lists and the repo does not have yet: lesson-dialogue lines (the characters in `tools/audio/cast.json`) **and** exam / listening tracks (narrator, man and woman archetypes), N5 and N4 together. Supersedes v4 (characters only), v3 and v2.

Every voice is cloned from a **saved reference** (`refs/` on Drive), so a session that resumes after a disconnect clones the same voice:
- characters: `refs/<id>@<rev>.wav + .json`, tuned by ear (as in v3 / v4), committed in `tools/audio/refs/`;
- exam voices: `refs/narrator`, `refs/man-<archetype>`, `refs/woman-<archetype>`, designed once by section 4 from the approved v2 prompts (narrator: preset Serena; women: preset Ono_Anna; men: voice design), then kept.

**Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Sections 5 (tuning) and 7 (review) do nothing unless you set them.

1. **Setup**: install, models, helpers, Drive (clips, refs and the input live in `MyDrive/jelly-audio-render`).
2. **Voice references**: upload `tools/audio/refs/*.wav + *.json` once. Later sessions skip this.
3. **Render input**: upload `tools/audio/render-input.json` once (saved to Drive; set `NEW_INPUT = True` after a re-export). Prints every voice: clips, rendered, to render, reference.
4. **Exam voices**: designs and saves the reference of every exam voice that has clips to render and none saved. Plays each new one.
5. **(Optional) Tune a voice**: a new character, a changed character voice, or another seed for an exam voice.
6. **Render**: every remaining clip, voice by voice, one progress bar with ETA.
7. **(Optional) Review / REDO**: play one voice's new clips, re-render the ones that drifted.
8. **Manifest + download**: verifies, writes `manifest.json`, downloads `jelly-audio-new.zip` (new mp3s, `manifest.json`, `refs/`).

**Disconnected?** Reconnect, **Run all** again: nothing is uploaded twice, finished clips are skipped, each clip is written atomically.

**Then, in the repo:** unzip into `.scratch/render/`, copy its mp3s into `audio/` and its `refs/` files into `tools/audio/refs/` (commit any new ref), `node tools/build-audio-manifest.js .scratch/render`, `node tools/build-sw.js`, `node .claude/hooks/run-tests.js`.

## 1. Setup

In [ ]:
#@title Install, models, helpers, Drive { display-mode: "form" }
!pip install -q -U qwen-tts soundfile tqdm

import os, gc, json, time, hashlib, zlib, subprocess, zipfile
import torch, numpy as np, soundfile as sf
from IPython.display import Audio, display
from google.colab import files as files_mod
from qwen_tts import Qwen3TTSModel
from tqdm.auto import tqdm
print(torch.cuda.get_device_name(0))

def load(name):  # T4: float16 (no bfloat16), no flash-attn
    return Qwen3TTSModel.from_pretrained(name, device_map="cuda:0", dtype=torch.float16)

_models = {}
def model(kind):  # loaded on first use: "design" / "custom" only to make a reference, "base" clones every line
    ids = {"custom": "Qwen/Qwen3-TTS-12Hz-1.7B-CustomVoice",
           "design": "Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign",
           "base":   "Qwen/Qwen3-TTS-12Hz-1.7B-Base"}
    if kind not in _models: _models[kind] = load(ids[kind])
    return _models[kind]

def unload(*kinds):  # three 1.7B models are too much for a T4 at once: free the reference makers before rendering
    for k in kinds: _models.pop(k, None)
    gc.collect(); torch.cuda.empty_cache()

def design_ref(spec, text, seed):
    """The reference clip of a voice: a preset speaker with an instruction, or a designed voice. Not reproducible across sessions, so it is saved."""
    torch.manual_seed(seed)
    if spec["speaker"]:
        w, sr = model("custom").generate_custom_voice(text=text, language="Japanese", speaker=spec["speaker"], instruct=spec["voice"])
    else:
        w, sr = model("design").generate_voice_design(text=text, language="Japanese", instruct=spec["voice"])
    return np.asarray(w[0], dtype=np.float32), sr

def cloner(ref):
    """speak(text, seed) -> (wave, sr): the Base model cloning one fixed reference clip."""
    prompt = model("base").create_voice_clone_prompt(ref_audio=(ref["wave"], ref["sr"]), ref_text=ref["text"])
    def speak(text, s):
        torch.manual_seed(s)
        w, sr2 = model("base").generate_voice_clone(text=text, language="Japanese", voice_clone_prompt=prompt)
        return np.asarray(w[0], dtype=np.float32), sr2
    return speak

def loudness(w, target_db):
    """Scale a line so its speech (silence ignored) sits at target_db RMS; never clip."""
    active = w[np.abs(w) > 0.02 * np.abs(w).max()]
    g = 10 ** (target_db / 20) / (np.sqrt(np.mean(active ** 2)) + 1e-9)
    w = w * g
    peak = np.abs(w).max()
    return w * (0.97 / peak) if peak > 0.97 else w

# the values approved for v2 / v3 / v4
SEED      = 1234                       # base of each clip's seed: SEED + crc32(clipKey) % 100000; also the exam voices' reference seed (as v2)
TARGET_DB = -20
GAIN_DB   = {"F": 1, "M": 0, "N": 0}
clip_seed = lambda key: SEED + zlib.crc32(key.encode("utf-8")) % 100000
NARRATOR_VOICE = "Speak slowly and clearly in a calm, neutral tone like an exam recording."   # preset Serena (v2)
EXAM_REF_TEXT  = "おはようございます。今日は天気がいいので、駅まで歩いて行きます。"                   # v2's reference sentence
CHAR_REF_TEXT  = "おはようございます。今日は天気がいいので、駅まで歩いて行きます。いいえ、違いますよ。"   # v3 / v4's

def play(w, sr, role):
    display(Audio(loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), rate=sr))

def clip_key(role, arch, say, take=None):     # same rule as tools/build-audio-manifest.js; arch '<id>@<rev>' = a revised voice
    return role + "|" + ("" if role == "N" else arch) + "|" + say + ("#" + str(take) if take and take > 1 else "")  # '#<take>' = a re-render

def clip_name(key):
    return hashlib.sha1(key.encode("utf-8")).hexdigest()[:12] + ".mp3"

assert clip_name(clip_key("N", "", "いち")) == hashlib.sha1("N||いち".encode("utf-8")).hexdigest()[:12] + ".mp3"
assert clip_key("F", "nami@3", "はい。", 2) == "F|nami@3|はい。#2" and clip_key("M", "worker", "はい。", 1) == "M|worker|はい。"

# output folder on Google Drive, so a disconnect resumes
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/jelly-audio-render"
except Exception as e:
    print("Drive not mounted (", e, "): using /content, NOT resumable and saved references are lost on disconnect")
    OUT = "/content/jelly-audio-render"
REFS = os.path.join(OUT, "refs")       # voice references: <voice id>.wav + .json
os.makedirs(REFS, exist_ok=True)
for d in (OUT, REFS):                  # leftovers of an interrupted write
    for f in os.listdir(d):
        if f.endswith(".tmp"): os.remove(os.path.join(d, f))

ref_file   = lambda rid, ext: os.path.join(REFS, rid + ext)
has_ref    = lambda rid: os.path.exists(ref_file(rid, ".json"))
on_disk    = lambda: {f for f in os.listdir(OUT) if f.endswith(".mp3")}
saved_refs = lambda: sorted(f[:-5] for f in os.listdir(REFS) if f.endswith(".json"))

def write_mp3(w, sr, role, name):      # loudness + MP3, written to a temp file and renamed (no half clips)
    sf.write("/content/_tmp.wav", loudness(w, TARGET_DB + GAIN_DB.get(role, 0)), sr)
    tmp = os.path.join(OUT, name + ".tmp")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "/content/_tmp.wav", "-ac", "1", "-ar", "24000", "-b:a", "48k", "-f", "mp3", tmp], check=True)
    os.replace(tmp, os.path.join(OUT, name))

def load_ref(rid):
    with open(ref_file(rid, ".json"), encoding="utf-8") as f: meta = json.load(f)
    w, sr = sf.read(ref_file(rid, ".wav"), dtype="float32")
    return {**meta, "wave": w, "sr": sr}

def save_ref(rid, meta, w, sr):        # lossless wav + json, each written to a temp file and renamed
    sf.write(ref_file(rid, ".wav.tmp"), w, sr, format="WAV", subtype="FLOAT")
    os.replace(ref_file(rid, ".wav.tmp"), ref_file(rid, ".wav"))
    with open(ref_file(rid, ".json.tmp"), "w", encoding="utf-8") as f: json.dump(meta, f, ensure_ascii=False, indent=1)
    os.replace(ref_file(rid, ".json.tmp"), ref_file(rid, ".json"))

print("output:", OUT, "| saved references:", ", ".join(saved_refs()) or "none")

## 2. Voice references (once)
Select every `.wav` and `.json` in `tools/audio/refs/` (characters, and exam voices once committed). They are copied to Drive and stay there: when Drive already holds references this cell does nothing (set `RE_UPLOAD = True` to upload again, e.g. after committing a new one).

In [ ]:
RE_UPLOAD = False

if saved_refs() and not RE_UPLOAD:
    print("Already on Drive, nothing to upload:", ", ".join(saved_refs()))
else:
    print("Select all of tools/audio/refs/*.wav and *.json")
    for name, data in files_mod.upload().items():
        open(ref_file(os.path.splitext(os.path.basename(name))[0], os.path.splitext(name)[1]), "wb").write(data)
    print("Saved references:", ", ".join(saved_refs()))

## 3. Render input
Upload `tools/audio/render-input.json` (from `node tools/export-tracks.js`). It is kept on Drive, so a resumed session reuses it: **set `NEW_INPUT = True` after every re-export**. Prints every voice: clips, already rendered (in `audio/` or on Drive), to render, saved reference.

In [ ]:
NEW_INPUT = False   # True: upload a new render-input.json. False: reuse the copy on Drive (first run uploads anyway)

INPUT = os.path.join(OUT, "render-input.json")
if NEW_INPUT or not os.path.exists(INPUT):
    print("Upload tools/audio/render-input.json")
    data = next(iter(files_mod.upload().values()))
    open(INPUT + ".tmp", "wb").write(data); os.replace(INPUT + ".tmp", INPUT)
else:
    print("Reusing the render-input.json on Drive from", time.ctime(os.path.getmtime(INPUT)), "(re-exported since? set NEW_INPUT = True)")
with open(INPUT, encoding="utf-8") as f: DATA = json.load(f)
TRACKS, CHARS, HAVE = DATA["tracks"], DATA["characters"], set(DATA.get("have", []))
MAN, WOMAN = DATA["manArchetypes"], DATA["womanArchetypes"]

rev_of = lambda c: CHARS[c].get("rev", 1)
ref_id = lambda c: f"{c}@{rev_of(c)}"          # a character's saved reference (its clipKey archetype is charArch in build-audio-manifest.js)

def arch_of(track, line):                      # a dialogue line names its character; the rest use the track's man / woman archetype
    return line.get("arch") or {"M": track["man"], "F": track["woman"]}.get(line["role"]) or ""

def voice_of(role, arch):                      # the voice a clip is cloned from = its saved reference's name
    if role == "N": return "narrator"
    c = arch.split("@")[0]
    if c in CHARS: return ref_id(c)
    return ("man-" if role == "M" else "woman-") + arch

def spec_of(v):                                # what a voice's reference is made from
    if v == "narrator":       return {"voice": NARRATOR_VOICE, "speaker": "Serena"}
    if v.startswith("man-"):   return {"voice": MAN[v[4:]], "speaker": None}
    if v.startswith("woman-"): return {"voice": WOMAN[v[6:]], "speaker": "Ono_Anna"}
    return {"voice": CHARS[v.split("@")[0]]["voice"], "speaker": None}

is_exam = lambda v: v == "narrator" or v.startswith(("man-", "woman-"))
role_of = lambda v: "N" if v == "narrator" else "M" if v.startswith("man-") else "F" if v.startswith("woman-") else CHARS[v.split("@")[0]]["gender"]
CLIPS, VOICES, LINES = {}, {}, {}             # clip name -> (role, arch, say, key, voice); voice -> clip names; voice -> lines
for t in TRACKS:
    for l in t["lines"]:
        a = arch_of(t, l); k = clip_key(l["role"], a, l["say"], l.get("take")); v = voice_of(l["role"], a)
        n = clip_name(k)
        if n not in CLIPS:
            CLIPS[n] = (l["role"], a, l["say"], k, v); VOICES.setdefault(v, []).append(n)
        LINES[v] = LINES.get(v, 0) + 1
left = lambda v, done: [n for n in VOICES.get(v, []) if n not in HAVE and n not in done]

def vid(x):                                    # a voice id from the table, or a bare character id
    if x in CHARS: return ref_id(x)
    assert x in VOICES, f"{x!r} is not a voice of this input. Choose from: {', '.join(VOICES)}"
    return x

def locked(v):                                 # the clone of the saved reference; refuses without one or when its prompt changed
    assert has_ref(v), f"No saved reference {v}: upload it (section 2); exam voice: run section 4; character: tune it (section 5)."
    ref, spec = load_ref(v), spec_of(v)
    assert ref["voice"] == spec["voice"] and ref.get("speaker") == spec["speaker"], (f"refs/{v} was made from another prompt than the input gives. "
        "Character: paste the saved prompt into cast.json with a new rev, or tune and save again. Exam voice: restore the archetype prompt, or delete refs/" + v + ".* before any of its clips ship.")
    return cloner(ref)

def status():
    done = on_disk()
    print(f"{'voice':22} {'lines':>5} {'clips':>5} {'rendered':>8} {'to render':>9}  reference")
    total = 0
    for v in sorted(VOICES, key=lambda v: (not is_exam(v), v)):
        n, todo = len(VOICES[v]), len(left(v, done))
        total += todo
        print(f"{v:22} {LINES[v]:>5} {n:>5} {n - todo:>8} {todo:>9}  {'saved' if has_ref(v) else '- MISSING'}")
    print(f"\n{len(TRACKS)} tracks, {len(CLIPS)} clips, {total} to render")
    return total
status();

## 4. Exam voices
Every exam voice (narrator, man / woman archetype) with clips to render and no saved reference gets one now: v2's prompt, reference sentence and seed, saved to Drive, played below. Already saved voices are left alone, so a resumed session keeps them. Not happy with one? Before rendering any of its clips, give it another seed in section 5.

In [ ]:
new = [v for v in VOICES if is_exam(v) and left(v, on_disk()) and not has_ref(v)]
for v in new:
    spec = spec_of(v)
    w, sr = design_ref(spec, EXAM_REF_TEXT, SEED)
    save_ref(v, {"id": v, **spec, "seed": SEED, "text": EXAM_REF_TEXT}, w, sr)
    print("Saved", v, "|", spec["speaker"] or "designed", "|", spec["voice"]); play(w, sr, role_of(v))
print(f"{len(new)} exam voice references made" + (": they go home in the zip of section 8 (refs/), commit them in tools/audio/refs/" if new else ""))
unload("design", "custom")

## 5. (Optional) Tune a voice
**Skip unless** a character's reference is MISSING (new character, or a changed voice), or an exam voice made in section 4 sounds wrong. With `TUNE = ""` both cells do nothing, so Run all is safe.

Tune: makes a reference from the prompt + `VOICE_SEED` and plays 3 of the voice's lines cloned from it. Change the seed (or, for a character, the prompt) and re-run until it sounds right. Save: writes `refs/<voice>` to Drive. A changed character prompt is saved as the next rev and the cell prints the `"rev"` and `"voice"` lines to paste into `tools/audio/cast.json`; then re-export and run section 3 with `NEW_INPUT = True`. An exam voice keeps its prompt (edit the archetype json for that, which renames nothing: delete its ref and re-design before any clip ships).

In [ ]:
TUNE       = ""     # a voice from the table ("yor", "man-worker", "narrator"); empty = skip this section
VOICE_SEED = 1234   # try 1, 2, 3, ...: other voices from the same prompt
VOICE      = ""     # characters only: empty = the prompt from cast.json; edit to try another

def samples(v):     # a short line, a long line, one starting with ……
    names = sorted(VOICES.get(v, []), key=lambda n: len(CLIPS[n][2]))
    if not names: return []
    pick = [names[0], names[-1]] + [n for n in names[1:-1] if CLIPS[n][2].startswith("……")][:1]
    return list(dict.fromkeys(pick))

if not TUNE:
    print("Skipped (TUNE is empty).")
else:
    T = vid(TUNE); spec = spec_of(T)
    assert not (VOICE and is_exam(T)), "An exam voice keeps its approved prompt: change the seed only."
    spec["voice"] = VOICE or spec["voice"]
    text = EXAM_REF_TEXT if is_exam(T) else CHAR_REF_TEXT
    w, sr = design_ref(spec, text, VOICE_SEED)
    CUR = {**spec, "seed": VOICE_SEED, "text": text, "wave": w, "sr": sr}
    print(T, "| prompt:", spec["voice"], "| seed:", VOICE_SEED, "(not saved yet)"); play(w, sr, role_of(T))
    speak = cloner(CUR)
    for n in samples(T):            # each with the seed the render will use for that clip
        role, arch, say, key, _ = CLIPS[n]
        w2, sr2 = speak(say, clip_seed(key))
        print(say); play(w2, sr2, role)

In [ ]:
SAVE_REFERENCE = False   # set True and run once the samples above sound right
OVERWRITE      = False   # True replaces an existing saved reference: only if none of its clips are rendered yet (else they no longer match)

if SAVE_REFERENCE and TUNE:
    c = T.split("@")[0] if not is_exam(T) else None
    changed = bool(c) and CUR["voice"] != CHARS[c]["voice"]
    rid = f"{c}@{rev_of(c) + 1}" if changed else T
    if has_ref(rid) and not OVERWRITE:
        print(rid, "is already saved: nothing written (set OVERWRITE = True to replace it)")
    else:
        meta = {"id": c, "rev": int(rid.split("@")[1])} if c else {"id": rid, "speaker": CUR["speaker"]}
        save_ref(rid, {**meta, "voice": CUR["voice"], "seed": CUR["seed"], "text": CUR["text"]}, CUR["wave"], CUR["sr"])
        print("Saved", rid, "to Drive (it goes home in the zip of section 8: commit it in tools/audio/refs/)")
        if changed:
            print(f"\nThe prompt changed. In tools/audio/cast.json, in the \"{c}\" entry, set these two fields:\n")
            print(f'"rev": {rid.split("@")[1]},')
            print(f'"voice": {json.dumps(CUR["voice"], ensure_ascii=False)}')
            print("\nThen run node tools/export-tracks.js and section 3 with NEW_INPUT = True.")
else:
    print("Nothing saved.")

## 6. Render
Every remaining clip, voice by voice (one clone prompt per voice), or only the voices in `RENDER`. Safe to stop and re-run: finished clips are skipped, each clip is written atomically. A voice without a saved reference is skipped (all mode) or refused (listed mode).

In [ ]:
RENDER = []        # e.g. ["yor", "narrator", "man-worker"]; empty = every voice with clips left

unload("design", "custom")
want = [vid(x) for x in RENDER] or [v for v in VOICES if left(v, on_disk())]
noref = [v for v in want if not has_ref(v)]
assert not (RENDER and noref), f"No saved reference for {noref}: upload it (section 2), run section 4 (exam) or tune it (section 5)."
if noref: print("SKIPPED, no saved reference:", ", ".join(noref), "\n")

done = on_disk()
jobs = [(v, left(v, done)) for v in want if v not in noref]
bar, t0, count = tqdm(total=sum(len(j) for _, j in jobs), desc="clips"), time.time(), 0
for v, todo in jobs:
    if not todo: continue
    speak = locked(v)
    bar.set_postfix_str(v)
    for n in todo:
        role, arch, say, key, _ = CLIPS[n]
        w, sr = speak(say, clip_seed(key))
        write_mp3(w, sr, role, n)
        bar.update(1); count += 1
bar.close()
print(f"{count} clips rendered in {(time.time() - t0) / 60:.1f} min ({(time.time() - t0) / max(count, 1):.1f} s per clip)\n")
status();

## 7. (Optional) Review / REDO
Set `REVIEW` to a voice: plays its clips rendered here (not the ones already in `audio/`), 50 at a time from `START`. If one drifted to another voice, put its number in `REDO` and run again: it re-renders from the same saved reference with another seed, under the same name (only before it ships; after that use `take`, see the README).

In [ ]:
REVIEW = ""        # a voice from the table, e.g. "nami" or "narrator"; empty = skip
START  = 0         # first clip number shown (50 per run)
REDO   = []        # clip numbers from the list below, e.g. [3, 17]
TRY    = 1         # bump to 2, 3, ... if a redone clip still drifts

if REVIEW:
    v = vid(REVIEW)
    MINE = sorted((n for n in VOICES[v] if n not in HAVE and n in on_disk()), key=lambda n: CLIPS[n][2])
    if REDO:
        speak = locked(v)
        for i in REDO:
            role, arch, say, key, _ = CLIPS[MINE[i]]
            w, sr = speak(say, clip_seed(key) + 7919 * TRY)
            write_mp3(w, sr, role, MINE[i])
            print("redone", i, say)
    print(f"{v}: {len(MINE)} new clips, showing {START}-{min(START + 50, len(MINE)) - 1}")
    for i, n in enumerate(MINE[START:START + 50], START):
        print(i, CLIPS[n][2]); display(Audio(os.path.join(OUT, n)))
else:
    print("Skipped (REVIEW is empty).")

## 8. Manifest and download
Needs every clip of every track in `audio/` or the output folder. Writes `manifest.json` and downloads `jelly-audio-new.zip`: the new mp3s, `manifest.json`, and `refs/` (every saved reference: copy them into `tools/audio/refs/`, git shows only the new ones).

In [ ]:
missing = [n for n in CLIPS if n not in HAVE and n not in on_disk()]
if missing: status()
assert not missing, f"{len(missing)} clips missing: run section 6 again"
manifest = {"version": 1, "tracks": {t["id"]: [clip_name(clip_key(l["role"], arch_of(t, l), l["say"], l.get("take"))) for l in t["lines"]] for t in TRACKS}}
assert all(len(manifest["tracks"][t["id"]]) == len(t["lines"]) for t in TRACKS)
with open(os.path.join(OUT, "manifest.json.tmp"), "w", encoding="utf-8") as f: json.dump(manifest, f, ensure_ascii=False, indent=1)
os.replace(os.path.join(OUT, "manifest.json.tmp"), os.path.join(OUT, "manifest.json"))

NEW = sorted(n for n in CLIPS if n not in HAVE)
ZIP = "/content/jelly-audio-new.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:   # mp3 does not compress
    for n in NEW: z.write(os.path.join(OUT, n), n)
    z.write(os.path.join(OUT, "manifest.json"), "manifest.json")
    for r in saved_refs():
        for ext in (".wav", ".json"): z.write(ref_file(r, ext), "refs/" + r + ext)
print(f"OK: {len(CLIPS)} clips accounted for, {len(NEW)} new, {len(manifest['tracks'])} tracks in manifest.json, {len(saved_refs())} refs")
print(f"zip: {os.path.getsize(ZIP) / 1e6:.1f} MB")
files_mod.download(ZIP)